In [2]:
import torch_geometric
from torch_geometric.datasets import Planetoid

In [4]:
dataset = Planetoid(root="data", name="cora")

Processing...
c:\Users\Prahlad\OneDrive\Documents\everything\EVs\.venv\Lib\site-packages\torch_geometric\io\planetoid.py:107: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  out = pickle.load(f, encoding='latin1')
c:\Users\Prahlad\OneDrive\Documents\everything\EVs\.venv\Lib\site-packages\torch_geometric\io\planetoid.py:107: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  out = pickle.load(f, encoding='latin1')
c:\Users\Prahlad\OneDrive\Documents\everything\EVs\.venv\Lib\site-packages\torch_geometric\io\planetoid.py:107: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  out = pickle

In [5]:
print("num of graphs:" , len(dataset))
print("num of classes: ", dataset.num_classes)
print("num of node features: " , dataset.num_node_features)
print("num of edge features" , dataset.num_edge_features)

num of graphs: 1
num of classes:  7
num of node features:  1433
num of edge features 0


In [ ]:
print(dataset.data)

In [13]:
import os.path as osp

import torch
from torch import nn, optim
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

In [9]:
data = dataset[0]

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv = SAGEConv(dataset.num_features, dataset.num_classes, aggr="max") # agg can be mean, add, max....
    
    def forward(self):
        x = self.conv(data.x, data.edge_index)
        return F.log_softmax(x, dim=1)

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, data = Net().to(device), data.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

In [25]:
def train():
    model.train()
    optimizer.zero_grad()
    F.nll_loss(model()[data.train_mask], data.y[data.train_mask]).backward()
    optimizer.step()
    
def test():
    model.eval()
    logits, accs = model(), []
    for _, mask in data('train_mask', 'val_mask', 'test_mask'):
        pred = logits[mask].max(1)[1]
        acc = pred.eq(data.y[mask]).sum().item() / mask.sum().item()
        accs.append(acc)
    
    return accs

In [27]:
best_val_acc = test_acc = 0
for epoch in range(1, 100):
    train()
    train_acc, val_acc, tmp_test_acc = test()
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        test_acc = tmp_test_acc
    log = "epoch: {:03d}, val: {:.4f}, test: {:.4f}"
    if epoch % 10 == 0:
        print(log.format(epoch, best_val_acc, test_acc))

epoch: 010, val: 0.7140, test: 0.7240
epoch: 020, val: 0.7180, test: 0.7260
epoch: 030, val: 0.7240, test: 0.7260
epoch: 040, val: 0.7240, test: 0.7260
epoch: 050, val: 0.7240, test: 0.7260
epoch: 060, val: 0.7240, test: 0.7260
epoch: 070, val: 0.7240, test: 0.7260
epoch: 080, val: 0.7240, test: 0.7260
epoch: 090, val: 0.7240, test: 0.7260
